# 进阶教程（七）：LLM 缓存与成本控制

> 相同的问题不要付两次钱。本讲用缓存 + 重试 + 计量三件套，把 LLM 成本压到最低。

## 本讲内容
1. 缓存原理：命中条件与两种实现（内存 / SQLite）
2. 缓存实测：同一问题二次调用的耗时与内容对比
3. `with_retry`：网络抖动自动重试（与 05 讲 fallbacks 的区别）
4. `usage_metadata`：token 级成本计量

# 0. 环境准备与运行说明

**前置要求：**
- 根目录 `.env` 已配置 `DEEPSEEK_API_KEY`（本教程用真实 DeepSeek，无本地降级）
- 已安装：`langchain>=1.3`、`langgraph>=1.2`、`langchain-classic`、`langchain-deepseek`
- 检索章节使用本地 `BAAI/bge-small-zh-v1.5` 嵌入（首次运行会下载模型，约 100MB）

**运行说明：**
- 按 cell 顺序执行；除标注外，每个示例消耗少量 API 额度（单次 < 0.01 元量级）
- 本教程面向已学完 `advanced_tutorial/01-06` 的开发者
- 各讲结尾 FAQ 汇总了基于 langchain 1.3.14 实测的导入路径与坑

In [1]:

# ========== 0. 初始化（每个 notebook 第一格） ==========
import os, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=DeprecationWarning)

# HF 镜像必须先于任何 langchain/huggingface 导入设置
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")

ROOT = Path.cwd().parent  # advanced_tutorial 的上一级 = 项目根目录
sys.path.insert(0, str(ROOT))
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

assert os.getenv("DEEPSEEK_API_KEY"), "请先在根目录 .env 配置 DEEPSEEK_API_KEY"

from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(model="deepseek-chat", temperature=0.2)
print("模型就绪:", model.__class__.__name__)


模型就绪: ChatDeepSeek


## 1. 缓存原理

LangChain 的 LLM 缓存以**「完整请求」为单位**：当一条消息序列与历史某次调用
**逐字一致**时，直接返回缓存结果，不再发请求。

两个关键点：
- **命中 = 输入完全相同**。改一个字、换一个参数，缓存即失效。
- 缓存是**实例/全局**行为，与具体模型无关，任何实现了 `BaseChatModel` 的模型都能用。

### 1.1 内存缓存：InMemoryCache

`set_llm_cache` 设全局缓存，所有模型自动生效：

In [3]:

import time
from langchain_core.caches import InMemoryCache
from langchain_core.globals import set_llm_cache

set_llm_cache(InMemoryCache())   # 全局缓存：进程内有效

q = "用一句话解释什么是向量数据库"

t0 = time.time()
r1 = model.invoke(q)
t1 = time.time() - t0

t0 = time.time()
r2 = model.invoke(q)             # 完全相同的请求 → 命中缓存
t2 = time.time() - t0

print(f"第一次 {t1:.2f}s | 第二次 {t2:.4f}s")
print("内容一致:", r1.content == r2.content)

第一次 2.13s | 第二次 0.0010s
内容一致: True


**预期输出**（大意）：
```
第一次 2.31s | 第二次 0.0001s
内容一致: True
```

第二次几乎零耗时——这就是缓存的价值。注意 `InMemoryCache` 进程重启即失效。

### 1.2 持久化缓存：SQLiteCache

生产环境要跨进程、跨重启复用，用 SQLite 落地到磁盘。
**注意导入路径**：`SQLiteCache` 在 `langchain_community.cache`（不在 `langchain_core.caches`，后者只有 `InMemoryCache`）：

In [4]:

from langchain_community.cache import SQLiteCache
from langchain_core.globals import set_llm_cache

set_llm_cache(SQLiteCache(database_path=".llm_cache.db"))

q2 = "用一句话解释什么是 Agent"

t0 = time.time()
model.invoke(q2)
t1 = time.time() - t0

t0 = time.time()
model.invoke(q2)                 # 命中磁盘缓存
t2 = time.time() - t0

print(f"第一次 {t1:.2f}s | 第二次 {t2:.4f}s")
print("缓存文件已生成:", Path(".llm_cache.db").exists())

第一次 1.28s | 第二次 0.0161s
缓存文件已生成: True


D:\PycharmProjects\my_langchain_demo\.venv\Lib\site-packages\langchain_community\cache.py:265: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(row[0]) for row in rows]


### 1.3 实例级缓存

不想全局污染时，直接给某个模型实例挂 `cache` 属性：

In [5]:

from langchain_core.caches import InMemoryCache

m = ChatDeepSeek(model="deepseek-chat", temperature=0.2)
m.cache = InMemoryCache()          # 只对这个实例生效
print("实例缓存已挂载:", m.cache is not None)

实例缓存已挂载: True


## 2. with_retry：网络抖动自动重试

缓存解决"重复请求"，重试解决"偶发失败"（429 限流、网络抖动）。
`with_retry` 基于 tenacity，指数退避自动重试。与 05 讲 `with_fallbacks` 的区别：

| 机制 | 解决什么 | 语义 |
|---|---|---|
| `with_retry` | 偶发失败 | **同一链路**重试 N 次 |
| `with_fallbacks` | 持续不可用 | **切换**到备用链路 |

In [ ]:

from langchain_core.runnables import RunnableLambda

class Flaky:
    calls = 0
    def invoke(self, x):
        Flaky.calls += 1
        if Flaky.calls <= 2:
            raise ConnectionError("模拟 429 限流")
        return f"第 {Flaky.calls} 次调用成功"

flaky = RunnableLambda(Flaky().invoke).with_retry(
    stop_after_attempt=3,
    retry_if_exception_type=(ConnectionError,),
)
print(flaky.invoke("test"))
print("总调用次数:", Flaky.calls)   # 2 次失败 + 1 次成功

真实模型直接：`model.with_retry(stop_after_attempt=3)`。
建议配合 `retry_if_exception_type` 只对限流类异常重试，参数错误等不重试。

## 3. usage_metadata：token 级成本计量

缓存之外，还要知道**每次到底花了多少 token**。
`AIMessage.usage_metadata` 直接给出输入/输出 token 数：

In [6]:

r = model.invoke("用 30 字解释什么是 RAG")
u = r.usage_metadata
print("回答:", r.content)
print("输入 tokens:", getattr(u, "input_tokens", None))
print("输出 tokens:", getattr(u, "output_tokens", None))
print("总计 tokens:", getattr(u, "total_tokens", None))

回答: RAG（检索增强生成）是一种AI技术，先检索外部知识库，再结合用户问题生成更准确、可溯源的回答。
输入 tokens: None
输出 tokens: None
总计 tokens: None


**要点**：`usage_metadata` 是模型返回的原始计量，可累加做成本看板。
生产上配合回调（04 讲的 Meter）可统一采集，无需在每个调用点手动取。

## 4. 常见问题（FAQ）

| 问题 | 原因 | 解决 |
|---|---|---|
| `cannot import name 'SQLiteCache' from 'langchain_core.caches'` | SQLiteCache 已迁出 core | 从 `langchain_community.cache` 导入 |
| 第二次调用还是慢 | 请求内容不完全一致（哪怕多一个空格） | 缓存以完整消息序列为 key，逐字比对 |
| 缓存命中但结果过期了 | 缓存没有 TTL（InMemory/SQLite 均为永久） | 需要时效性用 Redis 缓存 + 自己设过期 |
| 想对缓存做 key 定制 | 默认 key 不含 temperature | 自定义 `BaseCache` 子类覆写 key 生成 |
| InMemoryCache 重启就丢 | 内存实现 | 换 `SQLiteCache` / `RedisCache` 持久化 |